#### NCEP GFS Forecasts (1.00 degree grid) Data(atmosphere)을  JSON Format File 로 변환을 위한 분석 Note
###### NCEP NOMADS Version 2.3.29, November 2026


In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import sqlite3
import json

In [ ]:
file_path = "../data/download/gfs.t06z.pgrb2.1p00.f030"

In [ ]:
ds = xr.open_dataset(
    file_path,
    engine="cfgrib",
    backend_kwargs={"filter_by_keys": {"typeOfLevel": "isobaricInhPa", "level": 1000.}}
)

In [ ]:
ds.attrs

In [ ]:
gribEdition = ds.attrs['GRIB_edition']
centerName = ds.attrs['GRIB_centreDescription']
subcenter = ds.attrs['GRIB_subCentre']
refTime = pd.to_datetime(ds["time"].values ).strftime('%Y-%m-%dT%H:%M:%S.000Z')
forecastTime = int(f"{int(ds['step'].values / np.timedelta64(1, 'h')):02d}")
surface1Value = float(ds["isobaricInhPa"].item()) * 100


shape = ds.data_vars['t'].shape
numberPoints = ds.data_vars['t'].values.flatten().size
la2 = float(ds['t'].latitude.values.min())
la1 = float(ds['t'].latitude.values.max())
lo1 = float(ds['t'].longitude.values.min())
lo2 = float(ds['t'].longitude.values.max())
data = [ round(float(x), 5) for x in ds['t'].values.flatten() ]

In [ ]:
payload = [{"header":{"center":7,"centerName":centerName, "refTime":refTime,"parameterCategory":0,"parameterCategoryName":"Temperature","parameterNumber":0,"forecastTime":forecastTime,"surface1Type":100,"surface1TypeName":"Isobaric surface","surface1Value":surface1Value,"numberPoints":numberPoints,"winds":"true","nx":shape[1],"ny":shape[0],"basicAngle":0,"lo1":lo1,"la1":la1,"lo2":lo2,"la2":la2,"dx":1.0,"dy":1.0}, "data":data}]


In [ ]:
tupleRow = ('2026111406', 'air', 'temperature', 'isobaric', '1000hPa', json.dumps(payload))

In [ ]:
conn = sqlite3.connect('/Users/unchil/AndroidStudioProjects/OceanWaterInfo/oceanwater.sqlite')
cursor = conn.cursor()
cursor.execute( "INSERT INTO Earth_Current_Tbl (forecast_time, mode, type, surface, level, grib2json ) VALUES (?, ?, ?, ?, ?, ?) ",  tupleRow)
conn.commit()

#### parameter 정의
- **특수 키워드 `"wind"`** :
    - `parameterNumber: 2, 3` :
         - 바람 벡터 연산에 필요한 `UGRD`(동서 성분, parameterCategory 2 - parameterNumber 2)와 `VGRD`(남북 성분, parameterCategory 2 - parameterNumber 3)를 한 번에 동시 추출합니다.


- **주요 기상 변수 파라미터 코드 매핑**:

| parameterCategory| parameterCategoryName | parameterNumber | parameterNumberName | 설명 | `earth` 레이어 매핑 |
|:---|:---|:---|:---|:---|:---|
| **0**|**Temperature** | `0` | `TMP` | 기온 (Temperature, K) | 기온 (`temp`) |
| **0**|**Temperature** | `7` | `DPT` | 이슬점 온도 (Dew Point, K) | 이슬점 |
| **1**|**Moisture** | `1` | `RH` | 상대습도 (Relative Humidity, %) | 상대습도 (`relative_humidity`) |
| **1**|**Moisture** | `54` | `PWAT` | 가강수량 (Precipitable Water, $\text{kg/m}^2$) | 총 가강수량 (`total_precipitable_water`) |
| **2**|**Momentum** | `2`/`3` (wind) | `UGRD` & `VGRD` | U/V 풍속 벡터 성분 (m/s) | 바람 벡터 (`wind`) |
| **3**|**Mass** | `1` / `192` | `PRMSL` | 해면기압 (Pressure at MSL, Pa) | 해면기압 (`mean_sea_level_pressure`) |
| **6**|**Cloud** | `6` | `CWAT` | 구름수량 (Cloud Water, $\text{kg/m}^2$) | 총 구름수량 (`total_cloud_water`) |
| **7**|**Stability** | `6` | `CAPE` | 대기불안정도 (CAPE, J/kg) | 대기 대류 유효 전위 에너지 |




#### surface1Type 정의
- ex) "surface1Type":100,"surface1TypeName":"Isobaric surface","surface1Value":100000.0
- **surface1Value**
    - 지상~대류권 하부 (4개 층). 1000 hPa(약 100m), 925 hPa(약 800m), 850 hPa(약 1.5km), 700 hPa(약 3km)로 구성되어 하층 수증기 유입과 대기 불안정을 파악합니다.
    - 대류권 중·상부 (5개 층). 500 hPa(약 5.5km), 400 hPa(약 7.2km), 300 hPa(약 9.2km), 250 hPa(약 10.4km), 200 hPa(약 11.8km)로 구성되어 기압골, 기압능, 제트기류를 분석합니다.
    - 성층권 (7개 층). 150 hPa, 100 hPa(약 16.2km), 70 hPa, 50 hPa, 30 hPa, 20 hPa, 10 hPa(약 30km 이상)까지 이어지며 상층 대기 순환을 감시합니다.